# BFCL V4 — P06 smoke test (official harness)

Purpose: **actually run**, in a user-controlled GPU Colab session, one indexed family across base / missing function / missing parameter with a supported 4B model. This is a **smoke test**, not scientific evidence for a published result. This notebook was authored but **has not been run**. GPU must be enabled in Colab. Notebook pins the official Gorilla source commit; model revision is reported from resolved Hugging Face download information and should be frozen for full experiments. Data and model may require a substantial download.

2026-10-29 troubleshooting revision: BFCL runs in an isolated Python 3.12 environment, with streamed setup/inference/evaluation logs. This revision has not been executed on Colab. If a cell fails, download the corresponding file from the printed run directory under /content/P06_BFCL_Runs using Colab's Files panel. Do not continue to generation after setup failure.

Dependency repair: explicitly install soundfile in the Python 3.12 environment and test the complete BFCL CLI import chain before inference. User rerun reached CLI startup but failed before model inference; successful GPU execution remains unverified.

Tokenizer/server revision: pin Transformers 4.51.3 and validate the Qwen tokenizer through vLLM before generation. Launch vLLM separately using the official BFCL skip-server-setup option, choose FP16 on pre-Ampere GPUs, limit server context to 8192 and concurrency to one, and record the full serving profile. This resource-limited smoke test is not equivalent to a full-context leaderboard run. Context overflow must be reported as a runtime limitation. Model execution of this revision remains unverified.

Pre-run audit revision: corrected constraint newline; isolate each run under /content/P06_BFCL_Runs/<run-id>; validate requested inputs and references; block scoring if results are missing, duplicated or contain recorded inference exceptions; prevent retries from reusing failed records; preserve failure archives and stage status. Logs are now under the printed run directory, not the older shared log path. The final export cell can be run after a logged failure.


In [ ]:
import subprocess, sys, os, json, pathlib, datetime, platform
print('UTC', datetime.datetime.now(datetime.timezone.utc).isoformat())
print('Python', sys.version, 'Platform',platform.platform())
subprocess.run(['nvidia-smi'],check=True)



In [ ]:
from pathlib import Path
import shutil, hashlib, uuid, signal
run_id=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+uuid.uuid4().hex[:8]
run_root=Path('/content/P06_BFCL_Runs')/run_id
run_root.mkdir(parents=True,exist_ok=False)
run_status={'setup':'pending','selection':'pending','generation':'pending','evaluation':'pending'}
model='Qwen/Qwen3-4B-Instruct-2507-FC'
config={}
server_max_len=8192
root=Path('/content/gorilla')
if not root.exists():
    subprocess.run(['git','clone','--filter=blob:none','https://github.com/ShishirPatil/gorilla.git',str(root)],check=True)
commit='6ea57973c7a6097fd7c5915698c54c17c5b1b6c8'
subprocess.run(['git','-C',str(root),'checkout',commit],check=True)
work=root/'berkeley-function-call-leaderboard'
print('Commit:',subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'],text=True).strip())
# BFCL at this commit pins numpy==1.26.4 and vllm==0.8.5.
# Use an isolated interpreter; the Colab notebook kernel may remain Python 3.13.
log_dir=run_root/'logs'
log_dir.mkdir(exist_ok=True)
print('This run:',run_root,flush=True)
def export_outputs():
    # Each run uses a new staging directory, so old result/score files cannot leak in.
    bundle=run_root/'export'
    bundle.mkdir(exist_ok=True)
    for sub in ('logs','result','score'):
        src=run_root/sub
        if src.exists():
            shutil.copytree(src,bundle/sub,dirs_exist_ok=True)
    for name in ('test_case_ids_to_generate.json','run_status.json'):
        src=run_root/name
        if src.exists(): shutil.copy2(src,bundle/name)
    manifest={'run_id':run_id,'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
              'bfcl_commit':commit,'model_id':model,'selected_ids':config,
              'status':run_status.copy(),'note':'Resource-limited smoke test, not a complete study','files':{}}
    for item in bundle.rglob('*'):
        if item.is_file() and item.name!='manifest.json':
            manifest['files'][str(item.relative_to(bundle))]=hashlib.sha256(item.read_bytes()).hexdigest()
    (bundle/'manifest.json').write_text(json.dumps(manifest,indent=2))
    archive=shutil.make_archive(str(run_root/'P06_BFCL_Smoke_Outputs'),'zip',bundle)
    print('Diagnostic/output archive:',archive,flush=True)
    return archive

def save_status():
    (run_root/'run_status.json').write_text(json.dumps(run_status,indent=2))

def run_logged(cmd,name,cwd=None,env=None,timeout=3600):
    import threading
    log_path=log_dir/(name+'.log')
    print('Running:',' '.join(map(str,cmd)),flush=True)
    proc=None
    timer=None
    timed_out=[]
    def stop_group():
        timed_out.append(True)
        if proc is not None and proc.poll() is None:
            try: os.killpg(proc.pid,signal.SIGKILL)
            except ProcessLookupError: pass
    try:
        with log_path.open('w') as out:
            proc=subprocess.Popen(list(map(str,cmd)),cwd=cwd,env=env,
                stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
            timer=threading.Timer(timeout,stop_group)
            timer.daemon=True
            timer.start()
            for line in proc.stdout:
                print(line,end='',flush=True)
                out.write(line)
                out.flush()
            rc=proc.wait()
        if timed_out: raise TimeoutError(f'{name} exceeded {timeout} seconds; see {log_path}')
        if rc: raise subprocess.CalledProcessError(rc,cmd)
    except BaseException as exc:
        if proc is not None and proc.poll() is None:
            try: os.killpg(proc.pid,signal.SIGKILL)
            except ProcessLookupError: pass
            proc.wait()
        run_status[name]='failed: '+type(exc).__name__
        save_status()
        print('Saved failure log:',log_path,flush=True)
        export_outputs()
        raise
    finally:
        if timer is not None: timer.cancel()
    return log_path

run_logged([sys.executable,'-m','pip','install','uv'], 'install_uv')
uv_cmd=[sys.executable,'-m','uv']
run_logged(uv_cmd+['python','install','3.12'], 'install_python312')
venv=Path('/content/bfcl-py312')
if not (venv/'bin/python').exists():
    run_logged(uv_cmd+['venv','--python','3.12','--seed',str(venv)], 'create_venv')
bfcl_python=venv/'bin/python'
bfcl_cli=venv/'bin/bfcl'
run_logged([bfcl_python,'-c','import sys; print(sys.version); assert sys.version_info[:2] == (3,12)'], 'verify_python')
# vLLM 0.8.5 uses tokenizer APIs present in Transformers 4.51.3.
# Apply the constraint during resolution so a future major version is not selected.
constraints=Path('/content/P06_BFCL_constraints.txt')
constraints.write_text('transformers==4.51.3\n')
run_logged([bfcl_python,'-m','pip','install','-c',str(constraints),'-e',str(work)+'[oss_eval_vllm]'], 'install_bfcl')
# Qwen-Agent imports soundfile at CLI startup, including for text-only runs.
run_logged([bfcl_python,'-m','pip','install','soundfile'], 'install_soundfile')
run_logged([bfcl_python,'-m','pip','check'], 'pip_check')
run_logged([bfcl_python,'-c','import numpy, torch, vllm; print("numpy", numpy.__version__); print("torch", torch.__version__); print("vllm", vllm.__version__); print("CUDA", torch.cuda.is_available())'], 'verify_imports')
# Exercise the complete CLI import chain before attempting GPU inference.
preflight_env=os.environ.copy()
preflight_env['PATH']=str(venv/'bin')+os.pathsep+preflight_env.get('PATH','')
preflight_env['BFCL_PROJECT_ROOT']=str(run_root)
preflight_env['P06_CONTEXT_CAP']=str(server_max_len)
run_logged([bfcl_python,'-c','import soundfile; from qwen_agent.llm import get_chat_model; import bfcl_eval.__main__; print("BFCL/Qwen-Agent imports OK; soundfile", soundfile.__version__)'], 'verify_cli_imports', cwd=work, env=preflight_env)
run_logged([bfcl_cli,'generate','--help'], 'verify_generate_cli', cwd=work, env=preflight_env)
run_logged([bfcl_cli,'evaluate','--help'], 'verify_evaluate_cli', cwd=work, env=preflight_env)
# Download tokenizer/config only, then exercise vLLM's actual tokenizer wrapper.
tokenizer_check = """
import json, torch, transformers
from transformers import AutoTokenizer, AutoConfig
from vllm.transformers_utils.tokenizer import get_cached_tokenizer
model_id='Qwen/Qwen3-4B-Instruct-2507'
assert transformers.__version__ == '4.51.3', transformers.__version__
tok=AutoTokenizer.from_pretrained(model_id)
assert hasattr(tok, 'all_special_tokens_extended')
cached=get_cached_tokenizer(tok)
assert cached.encode('BFCL smoke test') == tok.encode('BFCL smoke test')
cfg=AutoConfig.from_pretrained(model_id)
print(json.dumps({'transformers':transformers.__version__, 'tokenizer':type(tok).__name__,
 'model_revision':getattr(cfg,'_commit_hash',None), 'gpu':torch.cuda.get_device_name(0),
 'capability':torch.cuda.get_device_capability(0), 'tokenizer_check':'PASS'}))
"""
run_logged([bfcl_python,'-c',tokenizer_check], 'verify_tokenizer', cwd=work, env=preflight_env)
run_logged([bfcl_python,'-m','pip','freeze'], 'pip_freeze')


run_status['setup']='passed'
save_status()


In [ ]:
assert run_status['setup']=='passed', 'Run setup successfully first.'
config={'multi_turn_base':['multi_turn_base_1'], 'multi_turn_miss_func':['multi_turn_miss_func_1'], 'multi_turn_miss_param':['multi_turn_miss_param_1']}
(run_root/'test_case_ids_to_generate.json').write_text(json.dumps(config,indent=2))
# Verify the official loader actually resolves all three requested records and references.
selection_check="""
import json, os
from transformers import AutoTokenizer
from bfcl_eval._llm_response_generation import build_handler
from bfcl_eval.constants.eval_config import TEST_IDS_TO_GENERATE_PATH
from bfcl_eval.utils import load_test_entries_from_id_file, load_ground_truth_entry
categories, entries=load_test_entries_from_id_file(TEST_IDS_TO_GENERATE_PATH)
requested=json.loads(TEST_IDS_TO_GENERATE_PATH.read_text())
expected={i for ids in requested.values() for i in ids}
assert len(entries)==len(expected)==3
assert {e['id'] for e in entries}==expected
tokenizer=AutoTokenizer.from_pretrained('Qwen/Qwen3-4B-Instruct-2507')
handler=build_handler('Qwen/Qwen3-4B-Instruct-2507-FC',0.001)
counts={}
for case in entries:
    prompt=handler._format_prompt(case['question'][0],case['function'])
    count=len(tokenizer.tokenize(prompt))
    counts[case['id']]=count
print('Initial prompt token counts:',counts)
assert max(counts.values())+4096<=int(os.environ['P06_CONTEXT_CAP']), 'Initial prompt plus BFCL generation budget exceeds server context cap.'
for category, ids in requested.items():
    refs={e['id']:e for e in load_ground_truth_entry(category)}
    for case_id in ids:
        case=next(e for e in entries if e['id']==case_id)
        assert case_id in refs
        assert len(case['question'])==len(refs[case_id]['ground_truth'])
print('PASS: three selected inputs and corresponding reference trajectories resolved.')
"""
run_logged([bfcl_python,'-c',selection_check],'verify_selection',cwd=work,env=preflight_env)
run_status['selection']='passed'
save_status()
print('Selected smoke-test IDs:',config)
print('Do not extrapolate results from one family.')


In [ ]:
# Start a resource-limited server via BFCL's official --skip-server-setup path.
# This avoids modifying BFCL source, prompts, decoding or the evaluator.
import signal, socket, time, urllib.request
model='Qwen/Qwen3-4B-Instruct-2507-FC'
model_checkpoint='Qwen/Qwen3-4B-Instruct-2507'
assert run_status['selection']=='passed', 'Validate selection first.'
os.environ['BFCL_PROJECT_ROOT']=str(run_root)
run_env=os.environ.copy()
run_env['PATH']=str(venv/'bin')+os.pathsep+run_env.get('PATH','')
# Remove stale remote endpoint overrides from earlier notebook experiments.
for key in ('REMOTE_OPENAI_BASE_URL','REMOTE_OPENAI_TOKENIZER_PATH','REMOTE_OPENAI_API_KEY'):
    run_env.pop(key, None)
run_env['LOCAL_SERVER_ENDPOINT']='127.0.0.1'
# Select a free local port rather than reuse a possibly occupied server.
with socket.socket() as sock:
    sock.bind(('127.0.0.1',0))
    port=sock.getsockname()[1]
run_env['LOCAL_SERVER_PORT']=str(port)
run_env['VLLM_USE_V1']='0'
run_env['PYTHONUNBUFFERED']='1'
capability=json.loads(subprocess.check_output(
    [str(bfcl_python),'-c','import json, torch; print(json.dumps(list(torch.cuda.get_device_capability(0))))'],
    text=True))
server_dtype='half' if capability[0] < 8 else 'bfloat16'
# server_max_len was also used by the initial-prompt preflight.
server_cmd=[str(venv/'bin/vllm'),'serve',model_checkpoint,
    '--host','127.0.0.1','--port',str(port),'--dtype',server_dtype,
    '--tensor-parallel-size','1','--gpu-memory-utilization','0.85',
    '--max-model-len',str(server_max_len),'--max-num-seqs','1',
    '--enforce-eager','--seed','0']
cmd=[str(bfcl_cli),'generate','--model',model,'--run-ids',
     '--backend','vllm','--skip-server-setup','--num-threads','1','--include-input-log','--allow-overwrite']
profile={'server_command':server_cmd,'generation_command':cmd,
         'dtype':server_dtype,'context_cap':server_max_len,'gpu_capability':capability,
         'note':'Resource-limited smoke profile. Overlong requests must be recorded as runtime errors, never silently truncated or treated as model failures.'}
(log_dir/'runtime_profile.json').write_text(json.dumps(profile,indent=2))
server_log=log_dir/'vllm_server.log'
print('Server command:', ' '.join(server_cmd), flush=True)
print('Server log:',server_log, flush=True)
server=None
run_status['generation']='running'
save_status()
try:
    with server_log.open('w') as out:
        server=subprocess.Popen(server_cmd,stdout=out,stderr=subprocess.STDOUT,
                                env=run_env,cwd=work,start_new_session=True)
        deadline=time.monotonic()+1800
        next_update=0
        while True:
            if server.poll() is not None:
                print(server_log.read_text(errors='replace')[-16000:])
                raise RuntimeError(f'vLLM exited with {server.returncode}; see {server_log}')
            try:
                with urllib.request.urlopen(f'http://127.0.0.1:{port}/v1/models',timeout=2) as response:
                    served=json.load(response)
                if model_checkpoint not in [item['id'] for item in served['data']]:
                    raise RuntimeError('Unexpected model served on selected port')
                print('Server ready.',flush=True)
                break
            except (OSError, ValueError):
                pass
            now=time.monotonic()
            if now>=deadline:
                print(server_log.read_text(errors='replace')[-16000:])
                raise TimeoutError(f'Server not ready after 30 minutes; see {server_log}')
            if now>=next_update:
                print('Waiting for model server; downloading/loading may take several minutes.',flush=True)
                next_update=now+30
            time.sleep(2)
        run_logged(cmd,'generate',cwd=work,env=run_env)
except BaseException as exc:
    run_status['generation']='failed: '+type(exc).__name__
    save_status()
    export_outputs()
    raise
finally:
    # Stop only the process group created by this cell, including worker children.
    if server is not None:
        try:
            os.killpg(server.pid,signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            server.wait(timeout=15)
        except subprocess.TimeoutExpired:
            pass
        try:
            os.killpg(server.pid,signal.SIGKILL)
        except ProcessLookupError:
            pass
        server.wait()

def check_generation_results():
    expected={case_id for ids in config.values() for case_id in ids}
    records=[]
    result_root=run_root/'result'/model.replace('/','_')
    for path in result_root.rglob('*_result.json'):
        records.extend(json.loads(line) for line in path.read_text().splitlines() if line.strip())
    observed=[item['id'] for item in records]
    errors=[item['id'] for item in records
            if item.get('traceback') or str(item.get('result','')).startswith('Error during inference:')]
    check={'expected_ids':sorted(expected),'observed_ids':observed,'inference_error_ids':errors}
    (log_dir/'result_integrity.json').write_text(json.dumps(check,indent=2))
    if set(observed)!=expected or len(observed)!=len(expected):
        raise RuntimeError('Missing, duplicate or unexpected result records; inspect result_integrity.json')
    if errors:
        raise RuntimeError('BFCL saved inference errors despite CLI success; inspect raw results and server log before scoring.')
try:
    check_generation_results()
    run_status['generation']='passed'
    save_status()
except BaseException as exc:
    run_status['generation']='failed postflight: '+type(exc).__name__
    save_status()
    export_outputs()
    raise


In [ ]:
assert run_status['generation']=='passed', 'Resolve generation/inference errors before evaluating.'
categories=','.join(config)
cmd=[str(bfcl_cli),'evaluate','--model',model,'--test-category',categories,'--partial-eval']
eval_env=os.environ.copy()
eval_env['BFCL_PROJECT_ROOT']=str(run_root)
eval_env['PATH']=str(venv/'bin')+os.pathsep+eval_env.get('PATH','')
run_status['evaluation']='running'
save_status()
try:
    run_logged(cmd,'evaluate',cwd=work,env=eval_env)
    assert any((run_root/'score').rglob('*.csv')), 'No score CSV produced.'
    run_status['evaluation']='passed'
    save_status()
except BaseException as exc:
    run_status['evaluation']='failed: '+type(exc).__name__
    save_status()
    export_outputs()
    raise


In [ ]:
# This cell also works after a logged failure; the manifest records incomplete stages.
from google.colab import files
files.download(export_outputs())
